In [0]:
from pyspark.sql import functions as F


In [0]:
BRONZE_TABLE = "interviews_dev.bronze.turbine_raw"
SILVER_TABLE = "interviews_dev.silver.turbine_clean"
QUARANTINE_TABLE = "interviews_dev.silver.turbine_quarantine"


In [0]:
bronze_df = spark.table(BRONZE_TABLE)

# 1. Remove rows with missing business keys.
# A row cannot be attributed to a turbine or time period without these fields.
key_valid_df = bronze_df.filter(
    F.col("turbine_id").isNotNull()
    & F.col("timestamp").isNotNull()
)

In [0]:
# 2. Define numeric columns that can be imputed.
numeric_columns = [
    "wind_speed",
    "wind_direction",
    "power_output",
]

# 3. Calculate averages from non-null values.
mean_row = (
    key_valid_df
    .select([
        F.avg(F.col(column)).alias(column)
        for column in numeric_columns
    ])
    .first()
)

mean_values = {
    column: mean_row[column]
    for column in numeric_columns
    if mean_row[column] is not None
}

print(mean_values)

In [0]:
silver_df = key_valid_df.fillna(mean_values)

# 5. Persist the cleaned Silver table.
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
quarantine_df = (
    bronze_df
    .filter(
        F.col("turbine_id").isNull()
        | F.col("timestamp").isNull()
    )
    .withColumn(
        "quarantine_reason",
        F.when(
            F.col("turbine_id").isNull()
            & F.col("timestamp").isNull(),
            F.lit("MISSING_TURBINE_ID_AND_TIMESTAMP")
        )
        .when(
            F.col("turbine_id").isNull(),
            F.lit("MISSING_TURBINE_ID")
        )
        .otherwise(
            F.lit("MISSING_TIMESTAMP")
        )
    )
    .withColumn("quarantined_at", F.current_timestamp())
)

# Write quarantine table in Unity Catalog
(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .saveAsTable(QUARANTINE_TABLE)
)